# SatQuery AI — Kaggle GPU Execution Notebook

This notebook provides **cloud GPU execution** for SatQuery AI:
- Uses **NVIDIA Tesla T4 (16GB VRAM)** on Kaggle.
- Fine-tunes **Qwen2.5-VL-3B-Instruct** using 4-bit QLoRA on BigEarthNet.
- Launches the **Gradio web interface** with `share=True` so you can use the complete system from your laptop browser.

### Step 1: Verify Kaggle GPU Accelerator

In [ ]:
!nvidia-smi

### Step 2: Clone Repository or Set Workspace
If you uploaded this code as a dataset or cloned from GitHub:

In [ ]:
import os

# Clone the repository if not already cloned
if not os.path.exists("SatQuery"):
    !git clone https://github.com/NizzyStark/SatQuery.git

# Change directory into SatQuery
%cd /kaggle/working/SatQuery

print("Current working directory:", os.getcwd())
!ls -la

### Step 3: Install Required Dependencies

In [ ]:
!pip install -q --upgrade pip
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers accelerate peft bitsandbytes qwen-vl-utils gradio rasterio scikit-image pillow pyyaml

### Step 4: Prepare Remote-Sensing Chat Dataset

In [ ]:
!python data/prepare_bigearthnet_txt.py --max-samples 5000 --output data/bigearthnet_chat.jsonl

### Step 5: Run QLoRA Fine-Tuning on GPU (T4)

In [ ]:
!python training/finetune_lora.py \
    --train-file data/bigearthnet_chat.jsonl \
    --output-dir models/lora_adapter \
    --epochs 1 \
    --batch-size 2 \
    --max-samples 500 \
    --use-4bit

### Step 6: Package the LoRA Adapter for Local Download
You can download this zip file directly from Kaggle's output pane to use on your laptop.

In [ ]:
!zip -r satquery_adapter.zip models/lora_adapter/
print("satquery_adapter.zip created successfully in Kaggle working directory!")

### Step 7: Launch the Full Interactive SatQuery Web App (Cloud-Hosted)
This creates a public Gradio URL (e.g. `https://xxxx.gradio.live`) so you can use SatQuery from your laptop's browser with Kaggle's GPU powering all inference.

In [ ]:
!python app.py --share